# Day 28: Debugging and Tracebacks

**Today's question:** when a program crashes, or quietly gives the wrong answer, how do we find the cause
*systematically* instead of guessing?

Several cells below fail **on purpose**. Before running a cell marked 🤔, predict what it will show.

## 1. Anatomy of a traceback

Three small functions that call each other, like a real program: `print_receipt` → `total_price` → `parse_price`.

In [ ]:
def parse_price(text):
    return float(text)


def total_price(price_texts):
    total = 0
    for text in price_texts:
        total += parse_price(text)
    return total


def print_receipt(price_texts):
    print("Total:", total_price(price_texts))

In [ ]:
print_receipt(["2.50", "1.20", "3.00"])

That works. Now one price was typed with a comma instead of a dot:

In [ ]:
print_receipt(["2.50", "1,20", "3.00"])

**Read a traceback from the bottom up:**

1. **Last line = WHAT went wrong:** `ValueError: could not convert string to float: '1,20'`.
2. **The block just above it = WHERE it happened:** inside `parse_price`, on the line `return float(text)`.
3. **Going further up = HOW we got there:** our cell called `print_receipt`, which called `total_price`,
   which called `parse_price`. "most recent call last" means the newest call is at the bottom.

The line that crashed is not always the line with the mistake. Here `parse_price` is fine; the **data** was bad.

## 2. 🤔 Your turn: read this one

In [ ]:
stock = {"apples": 12, "bread": 3, "milk": 8}


def quantity_of(item):
    return stock[item]


def needs_restock(item):
    return quantity_of(item) < 5

🤔 **Predict:** will `needs_restock("Bread")` crash? If so, with which error, and in which function?

In [ ]:
needs_restock("Bread")

`KeyError: 'Bread'`, raised in `quantity_of`, called from `needs_restock`. The key is `"bread"` with a small b.

## 3. The most common error types

| Error | Typical cause | Example |
|-------|---------------|---------|
| `SyntaxError` / `IndentationError` | Python cannot even read the code | missing `:` or `)`, wrong indentation |
| `NameError` | a name that does not exist (typo, not imported, cell not run) | `pirnt("hi")` |
| `TypeError` | operation on the wrong type, or wrong number of arguments | `"Total: " + 5` |
| `ValueError` | right type, impossible value | `int("twelve")` |
| `IndexError` | list position that does not exist | `[1, 2, 3][3]` |
| `KeyError` | dict key that does not exist | `stock["Bread"]` |
| `AttributeError` | method that the type does not have | `[1, 2].upper()` |
| `ZeroDivisionError` | dividing by zero, often `len([])` | `total / len(marks)` |
| `FileNotFoundError` | wrong file name or folder | `open("data/marks.txt")` |
| `ModuleNotFoundError` | import of a module that is not there | `import textutil` |

Three of them live:

In [ ]:
quantity = 5
print("Items in basket: " + quantity)

In [ ]:
names = ["Asha", "Bilal", "Sara"]
print(names[3])

In [ ]:
marks = [70, 85]
marks.upper()

## 4. Bugs that do NOT crash: print debugging

The worst bugs give a wrong answer with no error. This function should average the **last 3** quiz scores.

In [ ]:
def average_of_last(scores, n):
    recent = scores[-n:-1]
    return sum(recent) / len(recent)


print(average_of_last([55, 70, 80, 90], 3))
print(average_of_last([55, 70, 80, 90], 3) == 80.0)

No traceback, so we need to *look inside* the function. The quickest tool is a temporary `print`.

**New today:** inside an f-string, `{name=}` prints the variable's name **and** value. Text values even get
quotes, so you can see whether something is a string.

In [ ]:
def average_of_last(scores, n):
    recent = scores[-n:-1]
    print(f"DEBUG {scores=} {n=} {recent=}")
    return sum(recent) / len(recent)


print(average_of_last([55, 70, 80, 90], 3))

🤔 `recent` should be `[70, 80, 90]`. Where did the 90 go? `[-3:-1]` stops **before** the last item.
Fix it, and remove the debug `print` once the bug is gone.

In [ ]:
def average_of_last(scores, n):
    recent = scores[-n:]
    return sum(recent) / len(recent)


print(average_of_last([55, 70, 80, 90], 3) == 80.0)
print(average_of_last([90, 60], 2) == 75.0)

## 5. A debugging process (instead of guessing)

1. **Reproduce** the problem with the smallest input you can.
2. **Read** the traceback bottom-up (or write down *expected* vs *actual* output).
3. **Locate**: which function, which line, which value is wrong?
4. **Hypothesise**: "I think `recent` is missing the last score."
5. **Check** with a `print` or the debugger. Change **one thing at a time**.
6. **Fix**, then **re-run all your checks**, not just the one that failed.

## 6. The VS Code debugger with `buggy_grades.py`

`buggy_grades.py` (in this folder) prints a class report, and it has bugs. Importing it does not run it
(remember `if __name__ == "__main__":`), so we call its `main()` ourselves:

In [ ]:
import buggy_grades

buggy_grades.main()

This traceback goes through **a file**: `main` → `build_report` → `average`, then `ZeroDivisionError` in `average`.

Now switch to VS Code and follow the lesson steps:

| Action | How | What it does |
|--------|-----|--------------|
| Breakpoint | click left of a line number (red dot) | pause *before* that line runs |
| Start debugging | **F5** → *Python Debugger: Current File* | runs the file until a breakpoint |
| Step Over | **F10** | run the current line, stay in this function |
| Step Into | **F11** | go *inside* the function called on this line |
| Step Out | **Shift+F11** | finish this function, go back to the caller |
| Continue | **F5** | run until the next breakpoint |
| Stop | **Shift+F5** | end the debugging session |

While paused: the **Variables** panel shows every local variable, the **Call Stack** shows the same chain as the
traceback, and the **Debug Console** lets you type expressions like `sum(marks)` or `len(marks)`.

(Notebooks can be debugged too: the ▷ menu next to a cell has **Debug Cell**.)

## 7. Debugging habits that make things worse

| Habit | Why it hurts | Instead |
|-------|--------------|---------|
| Changing random things until it works | you never learn the cause and often add new bugs | hypothesis → check → fix |
| Reading only the first line of a traceback | the useful information is at the **bottom** | read bottom-up |
| Wrapping it in `try/except` to make the error disappear | the bug is hidden, not fixed | fix the cause; catch only expected errors |
| Fixing five things at once | you cannot tell which change helped | one change, then re-run |
| Leaving debug prints in | noisy output, confused users | delete them after the fix |

## 8. Real-world example: a discount that is slightly wrong

A shop says 15% off 19.99 should cost 16.99, but customers are charged 17.99.

In [ ]:
def price_after_discount(price, percent):
    discount = price * percent // 100
    return round(price - discount, 2)


print(price_after_discount(19.99, 15))

🤔 **Hypothesis:** the discount itself is wrong. Check it with a debug print:

In [ ]:
price = 19.99
percent = 15
print(f"{price * percent=}")
print(f"{price * percent // 100=}")
print(f"{price * percent / 100=}")

`//` is **floor division** (Day 03): it throws away the decimals, so the discount became 2.0 instead of about 3.
One character was the whole bug.

In [ ]:
def price_after_discount(price, percent):
    discount = price * percent / 100
    return round(price - discount, 2)


print(price_after_discount(19.99, 15) == 16.99)

## Recap

1. Read tracebacks **bottom-up**: last line = what, block above = where, further up = how we got there.
2. Know the common errors: `NameError`, `TypeError`, `ValueError`, `IndexError`, `KeyError`, `ZeroDivisionError`…
3. Wrong answers without errors: print the values with `f"{name=}"`, or pause with the debugger.
4. Debugger: breakpoint → F5 → Step Over (F10) / Step Into (F11) → watch the Variables panel.
5. Work systematically: reproduce → read → locate → hypothesise → check → fix → re-run all checks.

**Next time:** turning our `print(f(x) == expected)` checks into automatic **tests** with `assert` and pytest.